# 🎨 Kaggle All-in-One AI Studio (2x Tesla T4 16GB)
### Chuẩn OpenAI REST API trực tiếp qua Cloudflare Public Tunnel:
- 👁️ **VLM**: Qwen3.8 27B GGUF (UD-Q4_K_M + Vision mmproj trên Dual-GPU)
- 🎙️ **STT**: Whisper-large-v3-turbo (**Bản FULL FP16** trên GPU 0)
- 🔊 **TTS**: Kokoro-82M (**Bản FULL FP16** trên GPU 0)
- 🖼️ **GenImage**: FLUX.1-schnell (4-bit NF4 trên GPU 1)
- 🎬 **GenVideo**: Wan2.1 / Wan2.2 (Text-to-Video trên GPU 1)
- 🌐 **Endpoint**: Xuất trực tiếp URL Public chuẩn OpenAI ()

In [ ]:
# 1. Nạp biến môi trường tự động (Hỗ trợ Kaggle Secrets, file .env, hoặc cấu hình runtime)
import os, json

# Thiết lập tối ưu GPU & Cache
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HOME"] = "/tmp/hf_cache"

# Nạp Kaggle UserSecrets nếu được cấp quyền
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for key in ["KAGGLE_USERNAME", "KAGGLE_KEY", "HF_TOKEN", "NTFY_TOPIC"]:
        val = secrets.get_secret(key)
        if val:
            os.environ[key] = val
except Exception:
    pass

# Đọc file .env nếu có sẵn
for env_candidate in [".env", "/kaggle/working/Gen_Image-Video/kaggle/all-in-one/.env"]:
    if os.path.exists(env_candidate):
        try:
            with open(env_candidate, "r") as ef:
                for line in ef:
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, v = line.split("=", 1)
                        clean_val = v.strip().strip("\"").strip("\x27")
                        os.environ.setdefault(k.strip(), clean_val)
        except Exception:
            pass

# Tạo cấu hình ~/.kaggle/kaggle.json để Kaggle API / CLI hoạt động thông suốt
try:
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as kf:
            json.dump({
                "username": os.environ["KAGGLE_USERNAME"],
                "key": os.environ["KAGGLE_KEY"]
            }, kf)
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
except Exception:
    pass

print(f"✅ Môi trường cấu hình hoàn tất! User: {os.environ.get('KAGGLE_USERNAME', 'Chưa cấu hình')}")


In [ ]:
# 2. Đồng bộ mã nguồn Studio AI (GitHub + Live Payload)
import os, base64, io, tarfile

repo_dir = "/kaggle/working/Gen_Image-Video"
target_dir = "/kaggle/working/Gen_Image-Video/kaggle/all-in-one"

if not os.path.exists(repo_dir):
    !git clone https://github.com/nxc1802/Gen_Image-Video.git
else:
    !cd {repo_dir} && git pull origin main

%cd {target_dir}

payload_data = "..."
raw_tar = base64.b64decode(payload_data)
with tarfile.open(fileobj=io.BytesIO(raw_tar), mode="r:gz") as tar:
    tar.extractall(".")

print("✅ Toàn bộ module Studio AI phiên bản 3.5 đã đồng bộ thành công!")

In [ ]:
# 3. Cài đặt các thư viện cần thiết (Tối ưu tốc độ với Prebuilt Wheels)
import os, glob

# Thiết lập Swap file 12GB trên /tmp (73GB trống) làm lớp đệm bộ nhớ ảo an toàn
!fallocate -l 12G /tmp/swapfile 2>/dev/null && chmod 600 /tmp/swapfile 2>/dev/null && mkswap /tmp/swapfile 2>/dev/null && swapon /tmp/swapfile 2>/dev/null && echo "✅ Swap 12GB activated!" || true

wheels_dir = "/kaggle/input/kaggle-studio-wheels"
if os.path.exists(wheels_dir) and glob.glob(f"{wheels_dir}/*.whl"):
    print(f"⚡ Phát hiện bộ prebuilt wheels tại {wheels_dir} - Cài đặt offline siêu tốc...")
    # Cài đặt offline các gói lõi từ local wheel files
    !pip install -q --no-index --find-links={wheels_dir} diffusers kokoro openai-whisper bitsandbytes accelerate torchao transformers huggingface-hub
    !pip install -q gguf>=0.10.0
    
    # Kiểm tra wheel của llama-cpp-python (CUDA 12.1)
    if glob.glob(f"{wheels_dir}/*llama_cpp_python*.whl"):
        print("⚡ Cài đặt llama-cpp-python (CUDA 12.1) offline từ local wheel...")
        !pip install -q --no-index --find-links={wheels_dir} llama-cpp-python
    else:
        print("🌐 Wheel llama-cpp-python chưa có trong dataset - Tải bổ sung qua index URL...")
        !pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
        
    !pip install -q --no-index --find-links={wheels_dir} -r requirements.txt || pip install -q -r requirements.txt
else:
    print("🌐 Cài đặt thông thường qua PyPI (chưa mount prebuilt wheels dataset)...")
    !pip install -q -U torchao
    !pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
    !pip install -q -r requirements.txt
print("✅ Cài đặt thư viện hoàn tất!")


In [ ]:
# 4. Khởi động toàn bộ Studio và xuất Cloudflare Public URL
!while true; do     python -u main.py;     CODE=$?;     if [ $CODE -eq 0 ] || [ $CODE -eq 99 ]; then         echo "Clean exit requested ($CODE). Stopping notebook.";         break;     fi;     echo "Server exited with code $CODE, restarting in 3s...";     sleep 3; done
